# Natural Language Inference (Cosine Similarity + MMR + BM25 Retrieval)

In [5]:
import json
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch
import numpy as np
import plotly.express as px
import warnings
from openai import OpenAI
from rank_bm25 import BM25Okapi
from sklearn.metrics import accuracy_score, confusion_matrix
import time

# Suppress potential warnings
warnings.filterwarnings("ignore")

# ==========================================
# 1. DATA LOADING & SETUP
# ==========================================
print("--- Loading Datasets ---")

# Load Train Data
train_data = []
train_path = 'C:\\Users\\brand\\Downloads\\NTU FYP Dataset\\Natural Language Inference\\anli_v1.0\\R1\\train.jsonl'
with open(train_path, 'r', encoding='utf-8') as f:
    for line in f:
        train_data.append(json.loads(line))
df = pd.DataFrame(train_data)
print(f"Train Dataset loaded. Total records: {len(df)}")

# Load Test Data
test_data = []
test_path = 'C:\\Users\\brand\\Downloads\\NTU FYP Dataset\\Natural Language Inference\\anli_v1.0\\R1\\test.jsonl'
with open(test_path, 'r', encoding='utf-8') as f:
    for line in f:
        test_data.append(json.loads(line))
df_1 = pd.DataFrame(test_data)
print(f"Test Dataset loaded. Total records: {len(df_1)}")

# Setup Groq Client
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key="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"
)

# ==========================================
# 2. PRE-COMPUTATION (INDEXING)
# ==========================================
print("\n--- Pre-computing Indexes (Embeddings & BM25) ---")

# Load Model
print("Loading sentence-transformer model ('all-MiniLM-L6-v2')...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Prepare Text Corpus (Context + Hypothesis)
train_texts = [f"{row['context']} {row['hypothesis']}" for _, row in df.iterrows()]
train_data_store = df.to_dict('records')

# A. Generate Embeddings (For Cosine & MMR)
print("Generating embeddings for Cosine/MMR (this may take a minute)...")
train_embeddings = model.encode(train_texts, convert_to_tensor=True, show_progress_bar=True)

# B. Build BM25 Index (For BM25)
print("Tokenizing and building BM25 index...")
tokenized_corpus = [doc.split(" ") for doc in train_texts]
bm25 = BM25Okapi(tokenized_corpus)

print("✅ Indexing Complete.")

# ==========================================
# 3. HELPER FUNCTIONS
# ==========================================

def mmr_logic(query_embedding, candidate_embeddings, top_k=5, diversity=0.5):
    """Core MMR Algorithm"""
    # Calculate cosine similarity between query and all candidates
    candidate_similarity = util.cos_sim(query_embedding, candidate_embeddings)[0]
    
    candidate_indices = list(range(len(candidate_embeddings)))
    selected_indices = [int(torch.argmax(candidate_similarity))]
    candidate_indices.remove(selected_indices[0])
    
    for _ in range(top_k - 1):
        if not candidate_indices: break

        cand_to_query_sim = candidate_similarity[candidate_indices]
        cand_to_selected_sim = util.cos_sim(candidate_embeddings[candidate_indices], 
                                            candidate_embeddings[selected_indices])
        cand_to_selected_max_sim, _ = torch.max(cand_to_selected_sim, dim=1)
        
        mmr_score = (1 - diversity) * cand_to_query_sim - diversity * cand_to_selected_max_sim
        
        best_candidate_idx_relative = torch.argmax(mmr_score).item()
        best_candidate_idx = candidate_indices[best_candidate_idx_relative]
        
        selected_indices.append(best_candidate_idx)
        candidate_indices.remove(best_candidate_idx)
        
    return selected_indices

def get_exemplars(method, query_text, k=5):
    """Switch function to retrieve exemplars based on the method."""
    selected_indices = []
    
    # --- Method 1: Cosine Similarity (Pure Relevance) ---
    if method == 'cosine':
        query_embedding = model.encode(query_text, convert_to_tensor=True)
        # Semantic Search returns top-k by cosine similarity
        hits = util.semantic_search(query_embedding, train_embeddings, top_k=k)[0]
        selected_indices = [hit['corpus_id'] for hit in hits]

    # --- Method 2: MMR (Relevance + Diversity) ---
    elif method == 'mmr':
        query_embedding = model.encode(query_text, convert_to_tensor=True)
        # 1. Fetch larger pool (Top-20)
        hits = util.semantic_search(query_embedding, train_embeddings, top_k=20)[0]
        pool_indices = [hit['corpus_id'] for hit in hits]
        pool_embeddings = train_embeddings[pool_indices]
        
        # 2. Apply MMR Re-ranking
        mmr_relative_indices = mmr_logic(query_embedding, pool_embeddings, top_k=k, diversity=0.5)
        selected_indices = [pool_indices[i] for i in mmr_relative_indices]

    # --- Method 3: BM25 (Keyword Matching) ---
    elif method == 'bm25':
        tokenized_query = query_text.split(" ")
        scores = bm25.get_scores(tokenized_query)
        selected_indices = np.argsort(scores)[::-1][:k]
        
    return [train_data_store[i] for i in selected_indices]

# ==========================================
# 4. MAIN EXPERIMENT LOOP
# ==========================================
# We define the 3 methods to loop through
methods_to_test = ['cosine', 'mmr', 'bm25']

# Select Test Data (Reset sample once for consistency across methods if desired, 
# or sample inside loop. Here we sample once so all methods test on SAME data)
test_df_sample = df_1.sample(n=150, random_state=42).copy()
print(f"\nStarting evaluation loop on {len(test_df_sample)} test cases...")

for method in methods_to_test:
    print(f"\n" + "="*50)
    print(f"🚀 RUNNING EXPERIMENT: {method.upper()}")
    print("="*50)
    
    predictions = []
    
    for index, row in test_df_sample.iterrows():
        # A. Formulate Query
        current_pair_text = f"{row['context']} {row['hypothesis']}"
        
        # B. Retrieve Exemplars (Dynamic)
        exemplars = get_exemplars(method, current_pair_text, k=5)
        
        # C. Construct Prompt
        system_content = """You are an expert in Natural Language Inference. Classify the relationship: 'e' (entailment), 'n' (neutral), or 'c' (contradiction).

Here are similar examples to help you decide:

"""
        for ex in exemplars:
            system_content += f"Premise: {ex['context']}\nHypothesis: {ex['hypothesis']}\nLabel: {ex['label']}\n\n"
            
        system_content += "Based on the examples above, classify the user's provided pair. Only output the final label ('e', 'n', or 'c')."
        user_content = f"Premise: {row['context']}\nHypothesis: {row['hypothesis']}\nLabel:"

        # D. LLM Call (With Retry Logic)
        max_retries = 3
        attempts = 0
        success = False
        
        while attempts < max_retries and not success:
            try:
                response = client.chat.completions.create(
                    model="llama-3.1-8b-instant",
                    messages=[
                        {"role": "system", "content": system_content}, 
                        {"role": "user", "content": user_content}
                    ],
                    temperature=0,
                    max_tokens=5 
                )
                
                p_label = response.choices[0].message.content.strip().lower()
                if len(p_label) > 0 and p_label[0] in ['e', 'n', 'c']:
                    predictions.append(p_label[0])
                    # Optional: Print progress sparingly
                    if len(predictions) % 50 == 0:
                        print(f"  Processed {len(predictions)}/{len(test_df_sample)}...")
                    success = True
                else:
                    raise ValueError(f"Invalid output: {p_label}")

            except Exception as e:
                attempts += 1
                if attempts >= max_retries:
                    print(f"  Failed on item {index}: {e}")
                    predictions.append('error')
                else:
                    time.sleep(5) # Short sleep before retry

        # Rate limit politeness
        time.sleep(0.5)

    # ==========================================
    # 5. RESULTS & VISUALIZATION FOR CURRENT METHOD
    # ==========================================
    # Store predictions in a temporary column
    col_name = f'pred_{method}'
    test_df_sample[col_name] = predictions
    
    # Filter valid results
    results_df = test_df_sample[test_df_sample[col_name].isin(['e', 'n', 'c'])].copy()

    if not results_df.empty:
        # A. Accuracy
        acc = accuracy_score(results_df['label'], results_df[col_name])
        print(f"\n--- RESULTS FOR {method.upper()} ---")
        print(f"Accuracy: {acc:.2%}")
        
        # B. Confusion Matrix Plot
        labels = ['e', 'n', 'c']
        label_names = ['Entailment', 'Neutral', 'Contradiction']
        
        cm = confusion_matrix(results_df['label'], results_df[col_name], labels=labels)
        
        fig = px.imshow(cm,
                        labels=dict(x="Predicted Label", y="True Label", color="Count"),
                        x=label_names,
                        y=label_names,
                        text_auto=True,
                        color_continuous_scale='Blues',
                        title=f"Confusion Matrix: {method.upper()} Retrieval")
        
        fig.update_layout(title_x=0.5)
        
        # Save unique file for each method
        plot_filename = f'confusion_matrix_{method}.html'
        fig.write_html(plot_filename)
        print(f"✅ Plot saved to '{plot_filename}'")
        
    else:
        print(f"⚠️ No valid predictions for {method}")

# ==========================================
# 6. FINAL SAVE
# ==========================================
csv_filename = 'llm_predictions_NLI_dynamic_comparison.csv'
test_df_sample.to_csv(csv_filename, index=False)
print(f"\n🎉 All experiments complete. Consolidated results saved to '{csv_filename}'.")

--- Loading Datasets ---
Train Dataset loaded. Total records: 16946
Test Dataset loaded. Total records: 1000

--- Pre-computing Indexes (Embeddings & BM25) ---
Loading sentence-transformer model ('all-MiniLM-L6-v2')...
Generating embeddings for Cosine/MMR (this may take a minute)...


Batches:   0%|          | 0/530 [00:00<?, ?it/s]

Tokenizing and building BM25 index...
✅ Indexing Complete.

Starting evaluation loop on 150 test cases...

🚀 RUNNING EXPERIMENT: COSINE
  Processed 50/150...
  Processed 100/150...
  Processed 150/150...

--- RESULTS FOR COSINE ---
Accuracy: 45.33%
✅ Plot saved to 'confusion_matrix_cosine.html'

🚀 RUNNING EXPERIMENT: MMR
  Processed 50/150...
  Processed 100/150...
  Processed 150/150...

--- RESULTS FOR MMR ---
Accuracy: 49.33%
✅ Plot saved to 'confusion_matrix_mmr.html'

🚀 RUNNING EXPERIMENT: BM25
  Processed 50/150...
  Processed 100/150...
  Processed 150/150...

--- RESULTS FOR BM25 ---
Accuracy: 49.33%
✅ Plot saved to 'confusion_matrix_bm25.html'

🎉 All experiments complete. Consolidated results saved to 'llm_predictions_NLI_dynamic_comparison.csv'.


# Commonsense Reasoning (Cosine Similarity + MMR + Euclidean Distance)

In [7]:
import json
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch
import numpy as np
import plotly.express as px
import warnings
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import euclidean_distances, cosine_similarity # Added cosine_similarity
import time
from openai import OpenAI

# Suppress warnings
warnings.filterwarnings("ignore")

print("Loading CommonsenseQA dataset from Hugging Face...")

# --- 1. Load the Dataset (Train & Validation) ---
splits = {
    'train': 'data/train-00000-of-00001.parquet', 
    'validation': 'data/validation-00000-of-00001.parquet'
}

# Read Parquet files
df_train = pd.read_parquet("hf://datasets/tau/commonsense_qa/" + splits["train"])
df_val = pd.read_parquet("hf://datasets/tau/commonsense_qa/" + splits["validation"])

print(f"Train dataset loaded. Total records: {len(df_train)}")
print(f"Validation (Test) dataset loaded. Total records: {len(df_val)}")

# --- 2. Helper Functions to Format Data ---
def format_choices(choices_dict):
    """Converts the choices dict {'label':['A'..], 'text':['Tx'..]} into a string."""
    labels = list(choices_dict['label'])
    texts = list(choices_dict['text'])
    
    formatted = []
    for l, t in zip(labels, texts):
        formatted.append(f"({l}) {t}")
    return " ".join(formatted)

def get_correct_text(row):
    """Extracts the text of the correct answer for embedding comparison."""
    labels = list(row['choices']['label'])
    try:
        idx = labels.index(row['answerKey'])
        return row['choices']['text'][idx]
    except:
        return ""

# Apply formatting
print("Preprocessing columns...")
df_train['formatted_choices'] = df_train['choices'].apply(format_choices)
# We don't strictly need correct answer text for Train to retrieve, but good for consistency
df_train['correct_answer_text'] = df_train.apply(get_correct_text, axis=1)

df_val['formatted_choices'] = df_val['choices'].apply(format_choices)
df_val['correct_answer_text'] = df_val.apply(get_correct_text, axis=1)


# --- 3. Generate Embeddings & Metrics ---
print("Loading sentence-transformer model ('all-MiniLM-L6-v2')...")
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Generating embeddings for Training Set (Exemplar Bank)...")
# We need Train embeddings to find similar examples (Few-Shot)
train_q_embeddings = model.encode(df_train['question'].tolist(), convert_to_tensor=True, show_progress_bar=True)
train_q_np = train_q_embeddings.cpu().numpy()

print("Generating embeddings for Validation Set (Test)...")
df = df_val.copy()
q_embeddings = model.encode(df['question'].tolist(), convert_to_tensor=True, show_progress_bar=True)
a_embeddings = model.encode(df['correct_answer_text'].tolist(), convert_to_tensor=True, show_progress_bar=True)

q_np = q_embeddings.cpu().numpy()
a_np = a_embeddings.cpu().numpy()

# --- Metric 1: Cosine Similarity (for Subset Selection) ---
print("Calculating Cosine Similarity...")
cosine_scores = util.cos_sim(q_embeddings, a_embeddings)
df['cosine_similarity'] = [cosine_scores[i][i].item() for i in range(len(df))]

# --- Metric 2: Euclidean Distance (for Subset Selection) ---
print("Calculating Euclidean Distance...")
dists = []
for i in range(len(q_np)):
    d = euclidean_distances(q_np[i].reshape(1, -1), a_np[i].reshape(1, -1))
    dists.append(d[0][0])
df['euclidean_distance'] = dists

# --- Subset Selection Logic (MMR) ---
def mmr_subset_selection(df, doc_embeddings, relevance_scores, top_n=150, lambda_param=0.7):
    """Selects top_n rows to maximize: Lambda * Relevance - (1-Lambda) * Diversity"""
    print(f"Performing MMR Subset Selection (Top {top_n})...")
    unselected_indices = list(range(len(df)))
    selected_indices = []
    
    while len(selected_indices) < top_n:
        remaining_indices = [i for i in unselected_indices if i not in selected_indices]
        
        if not selected_indices:
            best_idx = np.argmax([relevance_scores[i] for i in remaining_indices])
            selected_global_idx = remaining_indices[best_idx]
        else:
            # Simple MMR: Maximize relevance, Minimize max_sim to selected
            selected_embeds = doc_embeddings[selected_indices]
            curr_mmr_vals = []
            
            # Optimization: Batch calc similarity can be memory intensive, doing row-by-row for safety
            # For speed in this specific script, we accept the loop overhead or use torch/matrix ops
            # Using simple loop for clarity/safety as per original
            for idx in remaining_indices:
                cand_embed = doc_embeddings[idx].reshape(1, -1)
                rel = relevance_scores[idx]
                sims = cosine_similarity(cand_embed, selected_embeds)[0]
                max_sim = np.max(sims)
                mmr_score = (lambda_param * rel) - ((1 - lambda_param) * max_sim)
                curr_mmr_vals.append(mmr_score)
            
            best_local_idx = np.argmax(curr_mmr_vals)
            selected_global_idx = remaining_indices[best_local_idx]
            
        selected_indices.append(selected_global_idx)
    return df.iloc[selected_indices].copy().reset_index(drop=True)

# --- Dynamic Few-Shot Retrieval Logic ---
def get_exemplars(query_vec, method="Cosine", k=5):
    """
    Retrieves k exemplar indices from train_q_np based on the method.
    query_vec: numpy array of shape (1, embedding_dim)
    """
    if method == "Cosine":
        # Highest Cosine Similarity
        sims = cosine_similarity(query_vec, train_q_np)[0]
        # argsort is ascending, so take last k and reverse
        return sims.argsort()[-k:][::-1]
        
    elif method == "Euclidean":
        # Lowest Euclidean Distance
        dists = euclidean_distances(query_vec, train_q_np)[0]
        # argsort is ascending (low distance first)
        return dists.argsort()[:k]
        
    elif method == "MMR":
        # MMR Retrieval (Relevance = Cosine, Diversity = 1-Sim)
        # 1. Get Top 50 relevant candidates first to optimize speed
        sims = cosine_similarity(query_vec, train_q_np)[0]
        top_candidates_idx = sims.argsort()[-50:][::-1] 
        
        selected_idx = []
        candidates = list(top_candidates_idx)
        
        while len(selected_idx) < k:
            if not selected_idx:
                selected_idx.append(candidates[0])
                candidates.pop(0)
            else:
                selected_vecs = train_q_np[selected_idx]
                mmr_scores = []
                for cand in candidates:
                    cand_vec = train_q_np[cand].reshape(1, -1)
                    rel = sims[cand]
                    div_penalty = np.max(cosine_similarity(cand_vec, selected_vecs)[0])
                    score = 0.7 * rel - 0.3 * div_penalty
                    mmr_scores.append(score)
                
                best_mmr_idx = np.argmax(mmr_scores)
                selected_idx.append(candidates[best_mmr_idx])
                candidates.pop(best_mmr_idx)
        return selected_idx
    
    return []

print("Metrics calculation complete.")


# --- 5. Prepare Subsets for Testing ---
N_SAMPLES = 150 

# Subset 1: Cosine (Highest Similarity)
print(f"\nSelecting top {N_SAMPLES} by Cosine Similarity...")
df_cosine = df.sort_values('cosine_similarity', ascending=False).head(N_SAMPLES).copy()
df_cosine['method'] = 'Cosine'

# Subset 2: Euclidean (Lowest Distance)
print(f"Selecting top {N_SAMPLES} by Euclidean Distance...")
df_euclidean = df.sort_values('euclidean_distance', ascending=True).head(N_SAMPLES).copy()
df_euclidean['method'] = 'Euclidean'

# Subset 3: MMR (Relevance + Diversity)
df_mmr = mmr_subset_selection(df, q_np, df['cosine_similarity'].values, top_n=N_SAMPLES, lambda_param=0.7)
df_mmr['method'] = 'MMR'

datasets_to_test = [df_cosine, df_euclidean, df_mmr]

# --- LLM Client Setup ---
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key="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" 
)

# --- Prediction Loop with Dynamic Exemplars ---
all_results = []

for subset_df in datasets_to_test:
    method_name = subset_df['method'].iloc[0]
    print(f"\n--- Running LLM on Method: {method_name} ---")
    
    predictions = []
    count = 0
    
    for index, row in subset_df.iterrows():
        # 1. Get embedding for the current question
        # (We can re-embed just this one sentence to be safe/easy)
        q_emb = model.encode(row['question'], convert_to_tensor=False).reshape(1, -1)
        
        # 2. Retrieve Dynamic Exemplars using the SAME method
        exemplar_indices = get_exemplars(q_emb, method=method_name, k=5)
        
        # 3. Construct System Prompt
        system_content = "You are an expert in Commonsense Reasoning. Answer the multiple-choice question.\nSelect the single best answer ('A', 'B', 'C', 'D', or 'E').\n\nExamples:\n"
        
        for idx in exemplar_indices:
            ex = df_train.iloc[idx]
            system_content += f"Q: {ex['question']}\nC: {ex['formatted_choices']}\nA: {ex['answerKey']}\n\n"
        
        system_content += "Based on the examples, answer the user's question. Output ONLY the label."
        
        # 4. User Prompt
        user_content = f"Q: {row['question']}\nC: {row['formatted_choices']}\nA:"
        
        # 5. API Call
        max_retries = 3
        attempts = 0
        success = False
        pred_label = 'Error'
        
        while attempts < max_retries and not success:
            try:
                response = client.chat.completions.create(
                    model="llama-3.1-8b-instant",
                    messages=[{"role": "system", "content": system_content}, 
                              {"role": "user", "content": user_content}],
                    temperature=0, max_tokens=5 
                )
                pred = response.choices[0].message.content.strip().upper()
                
                valid = [lbl for lbl in ['A', 'B', 'C', 'D', 'E'] if lbl in pred]
                if valid:
                    pred_label = valid[0]
                    success = True
                else:
                    raise ValueError(f"Invalid: {pred}")
                    
            except Exception as e:
                attempts += 1
                if attempts < max_retries:
                    time.sleep(2) # Short sleep
            time.sleep(1) # Rate limit
        
        predictions.append(pred_label)
        count += 1
        
        if count % 10 == 0:
            print(f"  Processed {count}/{N_SAMPLES}...")
            
    subset_df['predicted_label'] = predictions
    all_results.append(subset_df)

print("\nPrediction complete for all methods.")


# --- 6. Comparative Results ---
comparison_data = []

for res_df in all_results:
    method = res_df['method'].iloc[0]
    
    # Filter valid
    valid_res = res_df[res_df['predicted_label'].isin(['A', 'B', 'C', 'D', 'E'])]
    
    if len(valid_res) > 0:
        acc = accuracy_score(valid_res['answerKey'], valid_res['predicted_label'])
        comparison_data.append({'Method': method, 'Accuracy': acc, 'Samples': len(valid_res)})
        print(f"Method: {method:<10} | Accuracy: {acc:.2%}")
    else:
        print(f"Method: {method:<10} | No valid predictions found.")

# Save Combined Results
final_df = pd.concat(all_results)
output_filename = 'llm_predictions_commonsense_dynamic_comparison.csv'
final_df.to_csv(output_filename, index=False)
print(f"\nDetailed comparison saved to '{output_filename}'")

Loading CommonsenseQA dataset from Hugging Face...
Train dataset loaded. Total records: 9741
Validation (Test) dataset loaded. Total records: 1221
Preprocessing columns...
Loading sentence-transformer model ('all-MiniLM-L6-v2')...
Generating embeddings for Training Set (Exemplar Bank)...


Batches:   0%|          | 0/305 [00:00<?, ?it/s]

Generating embeddings for Validation Set (Test)...


Batches:   0%|          | 0/39 [00:00<?, ?it/s]

Batches:   0%|          | 0/39 [00:00<?, ?it/s]

Calculating Cosine Similarity...
Calculating Euclidean Distance...
Metrics calculation complete.

Selecting top 150 by Cosine Similarity...
Selecting top 150 by Euclidean Distance...
Performing MMR Subset Selection (Top 150)...

--- Running LLM on Method: Cosine ---
  Processed 10/150...
  Processed 20/150...
  Processed 30/150...
  Processed 40/150...
  Processed 50/150...
  Processed 60/150...
  Processed 70/150...
  Processed 80/150...
  Processed 90/150...
  Processed 100/150...
  Processed 110/150...
  Processed 120/150...
  Processed 130/150...
  Processed 140/150...
  Processed 150/150...

--- Running LLM on Method: Euclidean ---
  Processed 10/150...
  Processed 20/150...
  Processed 30/150...
  Processed 40/150...
  Processed 50/150...
  Processed 60/150...
  Processed 70/150...
  Processed 80/150...
  Processed 90/150...
  Processed 100/150...
  Processed 110/150...
  Processed 120/150...
  Processed 130/150...
  Processed 140/150...
  Processed 150/150...

--- Running LLM on

# Sentiment Analysis (Cosine Similarity + MMR + TF-IDF (Lexical Similarity))

In [8]:
import json
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch
import numpy as np
import plotly.express as px
import warnings
import time
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI

# Suppress potential warnings
warnings.filterwarnings("ignore")

# --- 1. Load Data ---
print("Loading TweetEval (Emotion) dataset...")

splits = {
    'train': 'hf://datasets/cardiffnlp/tweet_eval/emotion/train-00000-of-00001.parquet',
    'test': 'hf://datasets/cardiffnlp/tweet_eval/emotion/test-00000-of-00001.parquet'
}

# A. Load "Memory Bank" (Train Data) - This is where we find our shots
train_df = pd.read_parquet(splits["train"])
train_df = train_df.rename(columns={'text': 'context'}) 

# B. Load "Target Data" (Test Data)
df_1 = pd.read_parquet(splits["test"])
df_1 = df_1.rename(columns={'text': 'context'})

# Map Labels
label_map = {0: 'anger', 1: 'joy', 2: 'optimism', 3: 'sadness'}

# Standardize: Select a FIXED set of 150 test samples to ensure fair comparison
# We want to see which selection method gives the best accuracy on the SAME data.
test_df = df_1.sample(n=150, random_state=42).copy().reset_index(drop=True)

print(f"Train dataset (Exemplar Pool) loaded: {len(train_df)} records")
print(f"Test dataset (Target) loaded: {len(test_df)} records")




# --- 2. Pre-compute Embeddings for Retrieval ---
print("Loading models and pre-computing embeddings...")
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# A. SBERT Embeddings (For Cosine & MMR)
print("  Generating SBERT embeddings...")
train_embeddings = sbert_model.encode(train_df['context'].tolist(), convert_to_tensor=True, show_progress_bar=True)
test_embeddings = sbert_model.encode(test_df['context'].tolist(), convert_to_tensor=True, show_progress_bar=True)

# B. TF-IDF Embeddings (For Lexical Similarity)
print("  Generating TF-IDF embeddings...")
tfidf_vectorizer = TfidfVectorizer()
# Fit on BOTH train and test to ensure they share the same vocabulary
full_corpus = train_df['context'].tolist() + test_df['context'].tolist()
tfidf_vectorizer.fit(full_corpus)

tfidf_train = tfidf_vectorizer.transform(train_df['context'])
tfidf_test = tfidf_vectorizer.transform(test_df['context'])

print("Embeddings ready.")



# --- 3. Retrieval Helper Function ---

def get_dynamic_exemplars(test_idx, method, n_shots=5, lambda_param=0.7):
    """
    Selects 5 exemplars from train_df based on the method relative to test_df[test_idx]
    """
    indices = []
    
    # --- METHOD 1: COSINE (Semantic Similarity) ---
    if method == 'Cosine':
        # Measure sim between this Test Tweet and ALL Train Tweets
        query_emb = test_embeddings[test_idx]
        cos_scores = util.cos_sim(query_emb, train_embeddings)[0] # shape (n_train,)
        
        # Pick Top-K
        top_results = torch.topk(cos_scores, k=n_shots)
        indices = top_results.indices.tolist()

    # --- METHOD 2: MMR (Balanced Diversity) ---
    elif method == 'MMR':
        query_emb = test_embeddings[test_idx]
        cos_scores = util.cos_sim(query_emb, train_embeddings)[0]
        
        # Optimization: Only run MMR on the top 50 Semantic matches (to save time)
        top_k_candidates = 50 
        top_results = torch.topk(cos_scores, k=top_k_candidates)
        candidate_indices = top_results.indices.tolist()
        candidate_embeddings = train_embeddings[candidate_indices]
        
        selected_indices = []
        
        # Calculate similarity matrix among candidates (for diversity term)
        cand_sim_matrix = util.cos_sim(candidate_embeddings, candidate_embeddings)
        
        for _ in range(n_shots):
            best_mmr = -np.inf
            best_idx_in_candidates = -1
            
            for i, real_idx in enumerate(candidate_indices):
                if real_idx in selected_indices: continue
                
                # Relevance (Similarity to Test Tweet)
                relevance = cos_scores[real_idx].item()
                
                # Diversity (Dissimilarity to already picked shots)
                if not selected_indices:
                    diversity = 0
                else:
                    # Find local indices of currently selected shots
                    sel_local = [candidate_indices.index(x) for x in selected_indices]
                    # Max sim to any existing shot
                    diversity = torch.max(cand_sim_matrix[i][sel_local]).item()
                
                mmr_score = (lambda_param * relevance) - ((1 - lambda_param) * diversity)
                
                if mmr_score > best_mmr:
                    best_mmr = mmr_score
                    best_idx_in_candidates = real_idx
            
            if best_idx_in_candidates != -1:
                selected_indices.append(best_idx_in_candidates)
        
        indices = selected_indices

    # --- METHOD 3: LEXICAL (TF-IDF Similarity) ---
    elif method == 'Lexical':
        # Measure TF-IDF Cosine Sim
        query_vec = tfidf_test[test_idx]
        # cosine_similarity returns (1, n_train) array
        lex_scores = cosine_similarity(query_vec, tfidf_train)[0]
        
        # Pick Top-K indices
        # We use argpartition for speed, then sort the top K
        unsorted_top_k = np.argpartition(lex_scores, -n_shots)[-n_shots:]
        # Sort these by score descending
        indices = unsorted_top_k[np.argsort(lex_scores[unsorted_top_k])[::-1]]

    # --- Construct Prompt Text ---
    # Retrieve the actual text and labels for the selected indices
    shots_text = ""
    for idx in indices:
        row = train_df.iloc[idx]
        shots_text += f"Tweet: {row['context']}\nLabel: {row['label']}\n\n"
        
    return shots_text



# --- 4. Setup Client ---
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key="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" 
)

# --- 5. Run Experiment on 3 Criteria ---
criteria_list = ['Cosine', 'MMR', 'Lexical']
results_summary = []

for criteria in criteria_list:
    print(f"\n==================================================")
    print(f" TESTING CRITERIA: {criteria} (Dynamic 5-Shot)")
    print(f"==================================================")
    
    predictions = []
    
    # Loop through the Fixed Test Set
    for i, (index, row) in enumerate(test_df.iterrows()):
        
        # A. RETRIEVE DYNAMIC EXEMPLARS
        # We find the 5 shots specifically for *this* test tweet (i)
        dynamic_shots = get_dynamic_exemplars(test_idx=i, method=criteria, n_shots=5)
        
        # B. CONSTRUCT PROMPT
        system_content = f"""You are an expert in Emotion Classification. Classify the tweet into:
0: anger
1: joy
2: optimism
3: sadness

Here are relevant examples:
{dynamic_shots}
Based on the examples above, classify the following tweet. Only output the final label ('0', '1', '2', or '3').
"""
        user_content = f"Tweet: {row['context']}\nLabel:"
        
        # C. LLM PREDICTION
        max_retries = 3
        attempts = 0
        success = False
        pred = -1
        
        while attempts < max_retries and not success:
            try:
                response = client.chat.completions.create(
                    model="llama-3.1-8b-instant",
                    messages=[
                        {"role": "system", "content": system_content}, 
                        {"role": "user", "content": user_content}
                    ],
                    temperature=0, max_tokens=5 
                )
                p_text = response.choices[0].message.content.strip()
                
                # Validation
                if len(p_text) > 0 and p_text[0] in ['0', '1', '2', '3']:
                    pred = int(p_text[0])
                    # PRINT EVERY LINE
                    print(f"  {i+1}/{len(test_df)} -> True: {row['label']}, Predicted: {pred}")
                    success = True
                else:
                    raise ValueError(f"Invalid output: {p_text}")
            except Exception as e:
                attempts += 1
                if attempts == max_retries: 
                    print(f"  {i+1}/{len(test_df)} -> Error: {e}")
                    pred = -1
                time.sleep(2)
        
        predictions.append(pred)
        time.sleep(1) # Rate limit protection

    # Save Predictions to column
    col_name = f'pred_{criteria}'
    test_df[col_name] = predictions
    
    # Calculate Accuracy
    valid_results = test_df[test_df[col_name] != -1]
    acc = accuracy_score(valid_results['label'], valid_results[col_name])
    results_summary.append({'Criteria': criteria, 'Accuracy': acc})
    
    # Save CSV
    test_df.to_csv(f'llm_prediction_SentimentAnalysis_dynamic_{criteria}.csv', index=False)
    print(f"  >> Accuracy for {criteria}: {acc:.2%}")

# --- Final Summary ---
print("\n=== Final Experiment Summary ===")
print(pd.DataFrame(results_summary))

Loading TweetEval (Emotion) dataset...
Train dataset (Exemplar Pool) loaded: 3257 records
Test dataset (Target) loaded: 150 records
Loading models and pre-computing embeddings...
  Generating SBERT embeddings...


Batches:   0%|          | 0/102 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

  Generating TF-IDF embeddings...
Embeddings ready.

 TESTING CRITERIA: Cosine (Dynamic 5-Shot)
  1/150 -> True: 3, Predicted: 3
  2/150 -> True: 0, Predicted: 0
  3/150 -> True: 0, Predicted: 0
  4/150 -> True: 3, Predicted: 3
  5/150 -> True: 0, Predicted: 0
  6/150 -> True: 1, Predicted: 1
  7/150 -> True: 0, Predicted: 0
  8/150 -> True: 0, Predicted: 3
  9/150 -> True: 0, Predicted: 0
  10/150 -> True: 0, Predicted: 0
  11/150 -> True: 1, Predicted: 1
  12/150 -> True: 1, Predicted: 1
  13/150 -> True: 1, Predicted: 1
  14/150 -> True: 0, Predicted: 0
  15/150 -> True: 2, Predicted: 3
  16/150 -> True: 3, Predicted: 3
  17/150 -> True: 0, Predicted: 0
  18/150 -> True: 1, Predicted: 2
  19/150 -> True: 0, Predicted: 0
  20/150 -> True: 1, Predicted: 1
  21/150 -> True: 2, Predicted: 2
  22/150 -> True: 0, Predicted: 0
  23/150 -> True: 3, Predicted: 3
  24/150 -> True: 3, Predicted: 3
  25/150 -> True: 2, Predicted: 3
  26/150 -> True: 0, Predicted: 0
  27/150 -> True: 1, Predicte

  76/150 -> True: 0, Predicted: 0
  77/150 -> True: 3, Predicted: 1
  78/150 -> True: 1, Predicted: 1
  79/150 -> True: 0, Predicted: 1
  80/150 -> True: 0, Predicted: 0
  81/150 -> True: 3, Predicted: 3
  82/150 -> True: 0, Predicted: 0
  83/150 -> True: 3, Predicted: 0
  84/150 -> True: 3, Predicted: 3
  85/150 -> True: 1, Predicted: 1
  86/150 -> True: 2, Predicted: 2
  87/150 -> True: 1, Predicted: 1
  88/150 -> True: 1, Predicted: 1
  89/150 -> True: 3, Predicted: 3
  90/150 -> True: 0, Predicted: 0
  91/150 -> True: 0, Predicted: 0
  92/150 -> True: 1, Predicted: 1
  93/150 -> True: 0, Predicted: 0
  94/150 -> True: 0, Predicted: 3
  95/150 -> True: 0, Predicted: 0
  96/150 -> True: 1, Predicted: 3
  97/150 -> True: 3, Predicted: 3
  98/150 -> True: 3, Predicted: 3
  99/150 -> True: 1, Predicted: 1
  100/150 -> True: 3, Predicted: 3
  101/150 -> True: 0, Predicted: 0
  102/150 -> True: 3, Predicted: 3
  103/150 -> True: 1, Predicted: 1
  104/150 -> True: 1, Predicted: 1
  105/150

# Information Extraction (Cosine Similarity + MMR + Jaccard Similarity)

In [9]:
import json
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch
import numpy as np
import time
import warnings
import ast
from openai import OpenAI

# Suppress potential warnings
warnings.filterwarnings("ignore")

# --- 1. DATA PREPROCESSING ---
def process_ie_data(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            entry = json.loads(line)
            
            # 1. Context = The full sentence
            context = entry.get('sentence', '')
            
            # 2. Hypothesis = String representation of NER entities
            hypothesis = str(entry.get('ner', []))
            
            # 3. Target = The actual relation list
            rels = entry.get('rel', [])
            target_text = str(rels)

            data.append({
                "context": context,
                "hypothesis": hypothesis,
                "target_text": target_text
            })
    return pd.DataFrame(data)

print("Loading datasets...")
# UPDATE PATHS IF NEEDED
train_path = r'C:\Users\brand\Downloads\NTU FYP Dataset\Information Extraction\train.jsonl'
test_path = r'C:\Users\brand\Downloads\NTU FYP Dataset\Information Extraction\test.jsonl'

# Load Full Train (Pool for Exemplars)
df_train = process_ie_data(train_path)
print(f"Train dataset loaded. Total records: {len(df_train)}")

# Load Test (Sample 150 for Evaluation)
df_test = process_ie_data(test_path).sample(n=150, random_state=42).copy().reset_index(drop=True)
print(f"Test dataset loaded (Random Sample). Total records: {len(df_test)}")


# --- 2. EMBEDDINGS & RETRIEVAL SETUP ---
print("\nLoading sentence-transformer model ('all-MiniLM-L6-v2')...")
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Generating embeddings for retrieval...")
# Embed Train (Context) for retrieval pool
train_embeddings = model.encode(df_train['context'].tolist(), convert_to_tensor=True, show_progress_bar=True)
# Embed Test (Context) for queries
test_embeddings = model.encode(df_test['context'].tolist(), convert_to_tensor=True, show_progress_bar=True)


# --- 3. RETRIEVAL FUNCTIONS ---

def get_cosine_exemplars(test_idx, k=5):
    """Selects top-k most semantically similar training examples."""
    query_emb = test_embeddings[test_idx]
    # Compute dot product (cosine sim)
    scores = util.cos_sim(query_emb, train_embeddings)[0]
    top_k = torch.topk(scores, k=k)
    return df_train.iloc[top_k.indices.cpu().numpy()]

def get_jaccard_exemplars(test_idx, k=5):
    """Selects top-k examples with highest lexical overlap (Jaccard)."""
    query_text = df_test.iloc[test_idx]['context']
    query_tokens = set(query_text.lower().split())
    
    # Optimization: Pre-filter with Cosine (top 200) to speed up Jaccard calculation
    # otherwise calculating Jaccard for 10k+ rows per query is too slow
    query_emb = test_embeddings[test_idx]
    pre_filter_scores = util.cos_sim(query_emb, train_embeddings)[0]
    top_candidates_indices = torch.topk(pre_filter_scores, k=200).indices.cpu().numpy()
    
    scores = []
    for idx in top_candidates_indices:
        train_text = df_train.iloc[idx]['context']
        train_tokens = set(train_text.lower().split())
        
        union = len(query_tokens.union(train_tokens))
        if union == 0: 
            score = 0
        else:
            score = len(query_tokens.intersection(train_tokens)) / union
        scores.append((score, idx))
    
    # Sort by score descending
    scores.sort(key=lambda x: x[0], reverse=True)
    best_indices = [x[1] for x in scores[:k]]
    return df_train.iloc[best_indices]

def get_mmr_exemplars(test_idx, k=5, lambda_param=0.6):
    """Selects examples balancing Relevance (Cosine) and Diversity."""
    query_emb = test_embeddings[test_idx]
    cos_scores = util.cos_sim(query_emb, train_embeddings)[0]
    
    # Optimization: Work with top 50 relevant candidates
    top_candidates = torch.topk(cos_scores, k=50)
    candidate_indices = top_candidates.indices.cpu().numpy().tolist()
    
    selected_indices = []
    for _ in range(k):
        if not selected_indices:
            best_idx = candidate_indices[0]
        else:
            best_score = -float('inf')
            best_idx = -1
            
            for idx in candidate_indices:
                relevance = cos_scores[idx].item()
                
                # Diversity: max similarity to already selected
                cand_emb = train_embeddings[idx].unsqueeze(0)
                sel_embs = train_embeddings[selected_indices]
                max_sim = torch.max(util.cos_sim(cand_emb, sel_embs)).item()
                
                # MMR Formula
                mmr = (lambda_param * relevance) - ((1 - lambda_param) * max_sim)
                if mmr > best_score:
                    best_score = mmr
                    best_idx = idx
        
        selected_indices.append(best_idx)
        candidate_indices.remove(best_idx)
        
    return df_train.iloc[selected_indices]


# --- 4. LLM PREDICTION LOOP (Dynamic Few-Shot) ---

# Client Setup
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key="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"
)

methods = ["Cosine", "Jaccard", "MMR"]
results = {}

print(f"\nStarting Prediction on {len(df_test)} test samples using {len(methods)} methods...")

for method in methods:
    print(f"\n==========================================")
    print(f"Running Method: {method}")
    print(f"==========================================")
    
    predictions = []
    
    for i, row in df_test.iterrows():
        # 1. Retrieve Exemplars dynamically
        if method == "Cosine":
            shots = get_cosine_exemplars(i, k=5)
        elif method == "Jaccard":
            shots = get_jaccard_exemplars(i, k=5)
        elif method == "MMR":
            shots = get_mmr_exemplars(i, k=5)
            
        # 2. Construct System Prompt with specific shots
        system_content = """You are an expert in Information Extraction. Your task is to extract relations from a sentence given the identified entities.
Output the relations in a Python list format: [['Entity1', 'Relation', 'Entity2']].
If no defined relation exists, output [].

Here are examples:
"""
        for _, shot in shots.iterrows():
            system_content += f"Sentence: {shot['context']}\nEntities: {shot['hypothesis']}\nOutput: {shot['target_text']}\n\n"
        
        system_content += """Based on the examples above, extract relations for the user's input. Only output the list."""
        
        # 3. User Input
        user_content = f"Sentence: {row['context']}\nEntities: {row['hypothesis']}\nOutput:"
        
        # 4. API Call with Retry Logic
        attempts = 0
        success = False
        pred_text = "error"
        
        while attempts < 3 and not success:
            try:
                response = client.chat.completions.create(
                    model="llama-3.1-8b-instant",
                    messages=[
                        {"role": "system", "content": system_content}, 
                        {"role": "user", "content": user_content}
                    ],
                    temperature=0, 
                    max_tokens=128
                )
                pred_text = response.choices[0].message.content.strip()
                if len(pred_text) > 0:
                    success = True
                else:
                    raise ValueError("Empty response")
            except Exception as e:
                attempts += 1
                if attempts == 3:
                    print(f"  Error on index {i}: {e}")
                time.sleep(2)
        
        predictions.append(pred_text)
        time.sleep(1) # Rate limit
        
        # 5. Print Progress every 10 rows
        if (i + 1) % 10 == 0:
            display_pred = (pred_text[:40] + '..') if len(pred_text) > 40 else pred_text
            print(f"  [Row {i+1}/{len(df_test)}] Prediction: {display_pred}")

    # Store results in the test dataframe
    df_test[f'pred_{method}'] = predictions
    # Save intermediate results
    df_test.to_csv(f'results_IE_{method}.csv', index=False)
    print(f"Completed {method}.")

print("\nAll predictions complete.")


# --- 5. EVALUATION (Fair Metrics) ---

def calculate_advanced_metrics(truth_list, pred_list, model):
    # 1. Semantic Scores (Batch)
    # Ensure no NaNs
    clean_truth = [str(t) for t in truth_list]
    clean_pred = [str(p) if p is not None else "" for p in pred_list]
    
    gt_embeddings = model.encode(clean_truth, convert_to_tensor=True, show_progress_bar=False)
    pred_embeddings = model.encode(clean_pred, convert_to_tensor=True, show_progress_bar=False)
    
    # Pairwise Cosine
    cos_scores = util.cos_sim(gt_embeddings, pred_embeddings)
    semantic_scores = [max(0, cos_scores[i][i].item()) for i in range(len(truth_list))]
    
    # 2. F1 Scores (Iterative)
    f1_scores = []
    for t_str, p_str in zip(truth_list, pred_list):
        try:
            truth_set = set(tuple(x) for x in ast.literal_eval(str(t_str)))
            try:
                pred_item = ast.literal_eval(str(p_str))
                if isinstance(pred_item, list):
                    pred_set = set(tuple(x) for x in pred_item if isinstance(x, (list, tuple)))
                else:
                    pred_set = set()
            except:
                pred_set = set()
                
            tp = len(truth_set.intersection(pred_set))
            fp = len(pred_set - truth_set)
            fn = len(truth_set - pred_set)
            
            if tp == 0:
                f1 = 0.0
            else:
                p = tp / (tp + fp)
                r = tp / (tp + fn)
                f1 = 2 * (p * r) / (p + r)
            f1_scores.append(f1)
        except:
            f1_scores.append(0.0)
            
    return np.mean(semantic_scores), np.mean(f1_scores)

print("\n--- Final Comparative Efficacy Results ---")

summary_data = []

for method in methods:
    col_name = f'pred_{method}'
    # Filter out absolute errors if any (kept as 'error' string)
    valid_df = df_test[df_test[col_name] != 'error'].copy()
    
    if len(valid_df) > 0:
        sem_score, f1_score = calculate_advanced_metrics(
            valid_df['target_text'].tolist(), 
            valid_df[col_name].tolist(), 
            model
        )
        print(f"Strategy: {method:15} | Semantic Score: {sem_score:.2%} | F1 Score: {f1_score:.2%}")
        summary_data.append({
            "Strategy": method, 
            "Semantic_Score": sem_score, 
            "F1_Score": f1_score
        })

# Save Final Summary
summary_df = pd.DataFrame(summary_data)
summary_df.to_csv("llm_prediction_InformationExtraction_dynamic.csv", index=False)
print("\nResults saved!")

Loading datasets...
Train dataset loaded. Total records: 5575
Test dataset loaded (Random Sample). Total records: 150

Loading sentence-transformer model ('all-MiniLM-L6-v2')...
Generating embeddings for retrieval...


Batches:   0%|          | 0/175 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]


Starting Prediction on 150 test samples using 3 methods...

Running Method: Cosine
  [Row 10/150] Prediction: [['convolution', 'Used-For', 'object det..
  [Row 20/150] Prediction: [['AlexNet', 'Compare-With', 'VGG 1 6']]
  [Row 30/150] Prediction: [['Cityscapes', 'Benchmark-For', 'semant..
  [Row 40/150] Prediction: [['Indian Pines', 'Benchmark-For', 'clas..
  [Row 50/150] Prediction: [['2D LSTM', 'Apply-To', 'image segmenta..
  [Row 60/150] Prediction: [['GCN', 'Used-For', 'HSI classification..
  [Row 70/150] Prediction: [['DEML', 'Part-Of', 'deeper neural netw..
  [Row 80/150] Prediction: [['classification algorithms', 'Used-For..
  [Row 90/150] Prediction: [['FlyingChairs', 'Compare-With', 'Sinte..
  [Row 100/150] Prediction: [['LSTM', 'Compare-With', 'BERT']]
  [Row 110/150] Prediction: [['Silla Jr. & Freitas', 'Proposed', 'fl..
  [Row 120/150] Prediction: [['deep learning', 'Used-For', 'computer..
  [Row 130/150] Prediction: [['BN', 'Synonym-Of', 'batch normalizati..
  [Row 140/1

# Summarisation (Cosine + MMR + BM25)

In [1]:
import json
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch
import numpy as np
from bert_score import score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import time
import warnings
from openai import OpenAI

# Suppress warnings
warnings.filterwarnings("ignore")

# --- 1. Load Data---
print("Loading DialogSum dataset from Hugging Face...")
base_url = "hf://datasets/knkarthick/dialogsum/"
splits = {'train': 'train.csv', 'validation': 'validation.csv', 'test': 'test.csv'}

# FIX: Added engine='python' and on_bad_lines='skip' to handle formatting errors
try:
    # Load Train Data (The "Knowledge Base" for shots)
    df_train = pd.read_csv(base_url + splits["train"], engine='python', on_bad_lines='skip')
    print(f"Train dataset loaded. Total records: {len(df_train)}")

    # Load Test Data (The "Target" to evaluate)
    df_test = pd.read_csv(base_url + splits["test"], engine='python', on_bad_lines='skip')
    print(f"Test dataset loaded. Total records: {len(df_test)}")
except Exception as e:
    print(f"Error loading data: {e}")
    # Fallback for older pandas versions
    df_train = pd.read_csv(base_url + splits["train"], engine='python', error_bad_lines=False)
    df_test = pd.read_csv(base_url + splits["test"], engine='python', error_bad_lines=False)

# Use a sample of the test set for evaluation (150 items)
test_df = df_test.sample(n=150, random_state=42).copy()
print(f"\nEvaluation set prepared: {len(test_df)} items from Test set.")


# --- 2. Pre-compute Retrieval Artifacts (on Train Data) ---
print("\n--- Pre-computing Indices for Retrieval ---")

# A. Sentence Transformer (for Cosine & MMR)
print("Loading sentence-transformer model ('all-MiniLM-L6-v2')...")
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Encoding Training Dialogues (Dense Index)...")
# We encode the dialogues from the TRAIN set to use as potential shots
train_embeddings = model.encode(df_train['dialogue'].tolist(), convert_to_tensor=True, show_progress_bar=True)

# B. TF-IDF (for Lexical Search)
print("Fitting TF-IDF Vectorizer (Sparse Index)...")
vectorizer = TfidfVectorizer(stop_words='english')
# Fit on Train dialogues
train_tfidf_matrix = vectorizer.fit_transform(df_train['dialogue'].astype(str))


# --- 3. Retrieval Functions ---

def get_few_shots(target_dialogue, method, n_shots=3):
    """
    Retrieves n_shots from df_train based on the target_dialogue and method.
    """
    # 1. METHOD: COSINE SIMILARITY
    if method == "Cosine":
        target_emb = model.encode(target_dialogue, convert_to_tensor=True)
        # Calculate similarity against all train docs
        scores = util.cos_sim(target_emb, train_embeddings)[0]
        # Get top k indices
        top_k_indices = torch.topk(scores, n_shots).indices.cpu().numpy()
        return df_train.iloc[top_k_indices]

    # 2. METHOD: TF-IDF (Complex/Lexical)
    elif method == "BM25_Complex":
        target_vec = vectorizer.transform([str(target_dialogue)])
        # Dot product (Sparse Similarity)
        scores = (train_tfidf_matrix * target_vec.T).toarray().flatten()
        # Get top k indices
        top_k_indices = np.argsort(scores)[-n_shots:][::-1]
        return df_train.iloc[top_k_indices]

    # 3. METHOD: MMR (Diversity)
    elif method == "MMR":
        target_emb = model.encode(target_dialogue, convert_to_tensor=True)
        
        # First, get top N candidates (e.g. 50) by Semantic Similarity to reduce computation
        cosine_scores = util.cos_sim(target_emb, train_embeddings)[0]
        top_50_indices = torch.topk(cosine_scores, 50).indices.cpu().numpy()
        
        # MMR Logic on these 50 candidates
        candidate_embeddings = train_embeddings[top_50_indices].cpu().numpy()
        target_emb_np = target_emb.cpu().numpy().reshape(1, -1)
        
        selected_indices_local = []
        candidate_idx_local = list(range(len(top_50_indices)))
        lambda_param = 0.6 
        
        for _ in range(n_shots):
            if not selected_indices_local:
                # First: Most relevant
                # Re-calculate sim just for candidates
                sims = cosine_similarity(candidate_embeddings, target_emb_np).flatten()
                best_local_idx = np.argmax(sims)
            else:
                # MMR: Lambda * Rel - (1-Lambda) * Redundancy
                current_candidates_emb = candidate_embeddings[candidate_idx_local]
                selected_emb = candidate_embeddings[selected_indices_local]
                
                # Relevance
                sim_to_query = cosine_similarity(current_candidates_emb, target_emb_np).flatten()
                
                # Redundancy (Max sim to any selected)
                sim_to_selected = cosine_similarity(current_candidates_emb, selected_emb)
                max_sim_to_selected = np.max(sim_to_selected, axis=1)
                
                mmr_scores = (lambda_param * sim_to_query) - ((1 - lambda_param) * max_sim_to_selected)
                best_in_candidates = np.argmax(mmr_scores)
                best_local_idx = candidate_idx_local[best_in_candidates]
            
            selected_indices_local.append(best_local_idx)
            candidate_idx_local.remove(best_local_idx)
            
        # Map local indices back to global DataFrame indices
        final_indices = top_50_indices[selected_indices_local]
        return df_train.iloc[final_indices]
    
    return pd.DataFrame()


def construct_dynamic_system_prompt(shots):
    """
    Constructs the system prompt string using the retrieved shots.
    """
    base_instruction = "You are an expert Summarizer. Provide a concise summary of the dialogue. Output ONLY the summary.\n\nHere are some examples:\n\n"
    
    examples_text = ""
    for _, row in shots.iterrows():
        examples_text += f"Dialogue: {row['dialogue']}\nSummary: {row['summary']}\n\n"
        
    final_instruction = base_instruction + examples_text + "Based on the examples above, summarize the user's provided dialogue."
    return final_instruction


# --- 4. Prediction Loop ---

# Setup Client
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key="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" 
)

methods_to_test = ["Cosine"]
final_results = {}

for method in methods_to_test:
    print(f"\n==================================================")
    print(f"Starting Prediction Run using Method: {method}")
    print(f"==================================================")
    
    current_preds = []
    
    # Iterate through the Test Set
    for i, (index, row) in enumerate(test_df.iterrows()):
        target_dialogue = row['dialogue']
        
        # 1. Retrieve Dynamic Exemplars
        try:
            shots = get_few_shots(target_dialogue, method, n_shots=3)
            # 2. Construct Prompt with these shots
            system_content = construct_dynamic_system_prompt(shots)
        except Exception as e:
            print(f"Error in retrieval: {e}")
            system_content = "You are an expert Summarizer. Output ONLY the summary."

        user_content = f"Dialogue: {target_dialogue}\nSummary:"
        
        # 3. Call LLM
        for attempt in range(3):
            try:
                response = client.chat.completions.create(
                    model="llama-3.1-8b-instant",
                    messages=[
                        {"role": "system", "content": system_content}, 
                        {"role": "user", "content": user_content}
                    ],
                    temperature=0,
                    max_tokens=60
                )
                pred = response.choices[0].message.content.strip()
                if pred:
                    current_preds.append(pred)
                    break
            except Exception as e:
                if attempt == 2: 
                    current_preds.append("error")
                    print(f"API Error: {e}")
                time.sleep(5) # Short sleep for retry
        
        time.sleep(5) # Rate limit courtesy
        
        # Print progress every 10 rows
        if (i + 1) % 10 == 0:
            print(f"  Processed {i + 1}/{len(test_df)} rows...")

    # Store results
    final_results[method] = test_df.copy()
    final_results[method]['predicted_summary'] = current_preds
    time.sleep(60)

print("\nAll predictions complete.")


# --- 5. Evaluate with BERTScore & Compare ---
print("\n--- Final Evaluation (BERTScore F1) ---")

summary_metrics = []

for method_name, result_df in final_results.items():
    # 1. Filter out explicit errors ('error' string)
    # 2. Drop rows where 'summary' or 'predicted_summary' is NaN/None
    valid_df = result_df[
        (result_df['predicted_summary'] != 'error') & 
        (result_df['summary'].notna()) & 
        (result_df['predicted_summary'].notna())
    ].copy()
    
    if not valid_df.empty:
        # 3. Ensure strictly string type (prevents NoneType error)
        cands = valid_df['predicted_summary'].astype(str).tolist()
        refs = valid_df['summary'].astype(str).tolist()
        
        # Calculate BERTScore
        try:
            P, R, F1 = score(cands, refs, lang="en", verbose=False)
            
            avg_f1 = F1.mean().item()
            avg_p = P.mean().item()
            
            print(f"{method_name}: F1 = {avg_f1:.4f} | Precision = {avg_p:.4f}")
            
            # Save individual CSVs
            filename = f"llm_predictions_summarisation_dynamic_{method_name}.csv"
            valid_df.to_csv(filename, index=False)
            print(f"  Saved to {filename}")
            
            summary_metrics.append({'Method': method_name, 'F1': avg_f1, 'Precision': avg_p})
            
        except Exception as e:
            print(f"Error calculating score for {method_name}: {e}")
    else:
        print(f"Skipping {method_name}: No valid predictions found after filtering.")

# Display Summary Table
if summary_metrics:
    metrics_df = pd.DataFrame(summary_metrics).sort_values('F1', ascending=False)
    print("\nComparison Table:")
    print(metrics_df)
else:
    print("No valid results to display.")

C:\Users\brand\anaconda3\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Loading DialogSum dataset from Hugging Face...
Train dataset loaded. Total records: 12460
Test dataset loaded. Total records: 1500

Evaluation set prepared: 150 items from Test set.

--- Pre-computing Indices for Retrieval ---
Loading sentence-transformer model ('all-MiniLM-L6-v2')...
Encoding Training Dialogues (Dense Index)...


Batches:   0%|          | 0/390 [00:00<?, ?it/s]

Fitting TF-IDF Vectorizer (Sparse Index)...

Starting Prediction Run using Method: Cosine
  Processed 10/150 rows...
  Processed 20/150 rows...
  Processed 30/150 rows...
  Processed 40/150 rows...
  Processed 50/150 rows...
  Processed 60/150 rows...
  Processed 70/150 rows...
  Processed 80/150 rows...
  Processed 90/150 rows...
  Processed 100/150 rows...
  Processed 110/150 rows...
  Processed 120/150 rows...
  Processed 130/150 rows...
  Processed 140/150 rows...
  Processed 150/150 rows...

All predictions complete.

--- Final Evaluation (BERTScore F1) ---


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Cosine: F1 = 0.9080 | Precision = 0.8969
  Saved to predictions_dynamic_Cosine.csv

Comparison Table:
   Method        F1  Precision
0  Cosine  0.908031   0.896867


In [1]:
import json
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch
import numpy as np
from bert_score import score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import time
import warnings
from openai import OpenAI

# Suppress warnings
warnings.filterwarnings("ignore")

# --- 1. Load Data---
print("Loading DialogSum dataset from Hugging Face...")
base_url = "hf://datasets/knkarthick/dialogsum/"
splits = {'train': 'train.csv', 'validation': 'validation.csv', 'test': 'test.csv'}

# FIX: Added engine='python' and on_bad_lines='skip' to handle formatting errors
try:
    # Load Train Data (The "Knowledge Base" for shots)
    df_train = pd.read_csv(base_url + splits["train"], engine='python', on_bad_lines='skip')
    print(f"Train dataset loaded. Total records: {len(df_train)}")

    # Load Test Data (The "Target" to evaluate)
    df_test = pd.read_csv(base_url + splits["test"], engine='python', on_bad_lines='skip')
    print(f"Test dataset loaded. Total records: {len(df_test)}")
except Exception as e:
    print(f"Error loading data: {e}")
    # Fallback for older pandas versions
    df_train = pd.read_csv(base_url + splits["train"], engine='python', error_bad_lines=False)
    df_test = pd.read_csv(base_url + splits["test"], engine='python', error_bad_lines=False)

# Use a sample of the test set for evaluation (150 items)
test_df = df_test.sample(n=150, random_state=42).copy()
print(f"\nEvaluation set prepared: {len(test_df)} items from Test set.")


# --- 2. Pre-compute Retrieval Artifacts (on Train Data) ---
print("\n--- Pre-computing Indices for Retrieval ---")

# A. Sentence Transformer (for Cosine & MMR)
print("Loading sentence-transformer model ('all-MiniLM-L6-v2')...")
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Encoding Training Dialogues (Dense Index)...")
# We encode the dialogues from the TRAIN set to use as potential shots
train_embeddings = model.encode(df_train['dialogue'].tolist(), convert_to_tensor=True, show_progress_bar=True)

# B. TF-IDF (for Lexical Search)
print("Fitting TF-IDF Vectorizer (Sparse Index)...")
vectorizer = TfidfVectorizer(stop_words='english')
# Fit on Train dialogues
train_tfidf_matrix = vectorizer.fit_transform(df_train['dialogue'].astype(str))


# --- 3. Retrieval Functions ---

def get_few_shots(target_dialogue, method, n_shots=3):
    """
    Retrieves n_shots from df_train based on the target_dialogue and method.
    """
    # 1. METHOD: COSINE SIMILARITY
    if method == "Cosine":
        target_emb = model.encode(target_dialogue, convert_to_tensor=True)
        # Calculate similarity against all train docs
        scores = util.cos_sim(target_emb, train_embeddings)[0]
        # Get top k indices
        top_k_indices = torch.topk(scores, n_shots).indices.cpu().numpy()
        return df_train.iloc[top_k_indices]

    # 2. METHOD: TF-IDF (Complex/Lexical)
    elif method == "BM25_Complex":
        target_vec = vectorizer.transform([str(target_dialogue)])
        # Dot product (Sparse Similarity)
        scores = (train_tfidf_matrix * target_vec.T).toarray().flatten()
        # Get top k indices
        top_k_indices = np.argsort(scores)[-n_shots:][::-1]
        return df_train.iloc[top_k_indices]

    # 3. METHOD: MMR (Diversity)
    elif method == "MMR":
        target_emb = model.encode(target_dialogue, convert_to_tensor=True)
        
        # First, get top N candidates (e.g. 50) by Semantic Similarity to reduce computation
        cosine_scores = util.cos_sim(target_emb, train_embeddings)[0]
        top_50_indices = torch.topk(cosine_scores, 50).indices.cpu().numpy()
        
        # MMR Logic on these 50 candidates
        candidate_embeddings = train_embeddings[top_50_indices].cpu().numpy()
        target_emb_np = target_emb.cpu().numpy().reshape(1, -1)
        
        selected_indices_local = []
        candidate_idx_local = list(range(len(top_50_indices)))
        lambda_param = 0.6 
        
        for _ in range(n_shots):
            if not selected_indices_local:
                # First: Most relevant
                # Re-calculate sim just for candidates
                sims = cosine_similarity(candidate_embeddings, target_emb_np).flatten()
                best_local_idx = np.argmax(sims)
            else:
                # MMR: Lambda * Rel - (1-Lambda) * Redundancy
                current_candidates_emb = candidate_embeddings[candidate_idx_local]
                selected_emb = candidate_embeddings[selected_indices_local]
                
                # Relevance
                sim_to_query = cosine_similarity(current_candidates_emb, target_emb_np).flatten()
                
                # Redundancy (Max sim to any selected)
                sim_to_selected = cosine_similarity(current_candidates_emb, selected_emb)
                max_sim_to_selected = np.max(sim_to_selected, axis=1)
                
                mmr_scores = (lambda_param * sim_to_query) - ((1 - lambda_param) * max_sim_to_selected)
                best_in_candidates = np.argmax(mmr_scores)
                best_local_idx = candidate_idx_local[best_in_candidates]
            
            selected_indices_local.append(best_local_idx)
            candidate_idx_local.remove(best_local_idx)
            
        # Map local indices back to global DataFrame indices
        final_indices = top_50_indices[selected_indices_local]
        return df_train.iloc[final_indices]
    
    return pd.DataFrame()


def construct_dynamic_system_prompt(shots):
    """
    Constructs the system prompt string using the retrieved shots.
    """
    base_instruction = "You are an expert Summarizer. Provide a concise summary of the dialogue. Output ONLY the summary.\n\nHere are some examples:\n\n"
    
    examples_text = ""
    for _, row in shots.iterrows():
        examples_text += f"Dialogue: {row['dialogue']}\nSummary: {row['summary']}\n\n"
        
    final_instruction = base_instruction + examples_text + "Based on the examples above, summarize the user's provided dialogue."
    return final_instruction



# Setup Client
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key="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" 
)


methods_to_test = ["MMR"]
final_results = {}

for method in methods_to_test:
    print(f"\n==================================================")
    print(f"Starting Prediction Run using Method: {method}")
    print(f"==================================================")
    
    current_preds = []
    
    # Iterate through the Test Set
    for i, (index, row) in enumerate(test_df.iterrows()):
        target_dialogue = row['dialogue']
        
        # 1. Retrieve Dynamic Exemplars
        try:
            shots = get_few_shots(target_dialogue, method, n_shots=3)
            # 2. Construct Prompt with these shots
            system_content = construct_dynamic_system_prompt(shots)
        except Exception as e:
            print(f"Error in retrieval: {e}")
            system_content = "You are an expert Summarizer. Output ONLY the summary."

        user_content = f"Dialogue: {target_dialogue}\nSummary:"
        
        # 3. Call LLM
        for attempt in range(3):
            try:
                response = client.chat.completions.create(
                    model="llama-3.1-8b-instant",
                    messages=[
                        {"role": "system", "content": system_content}, 
                        {"role": "user", "content": user_content}
                    ],
                    temperature=0,
                    max_tokens=60
                )
                pred = response.choices[0].message.content.strip()
                if pred:
                    current_preds.append(pred)
                    break
            except Exception as e:
                if attempt == 2: 
                    current_preds.append("error")
                    print(f"API Error: {e}")
                time.sleep(5) # Short sleep for retry
        
        time.sleep(5) # Rate limit courtesy
        
        # Print progress every 10 rows
        if (i + 1) % 10 == 0:
            print(f"  Processed {i + 1}/{len(test_df)} rows...")

    # Store results
    final_results[method] = test_df.copy()
    final_results[method]['predicted_summary'] = current_preds
    time.sleep(60)

print("\nAll predictions complete.")


# --- 5. Evaluate with BERTScore & Compare ---
print("\n--- Final Evaluation (BERTScore F1) ---")

summary_metrics = []

for method_name, result_df in final_results.items():
    # 1. Filter out explicit errors ('error' string)
    # 2. Drop rows where 'summary' or 'predicted_summary' is NaN/None
    valid_df = result_df[
        (result_df['predicted_summary'] != 'error') & 
        (result_df['summary'].notna()) & 
        (result_df['predicted_summary'].notna())
    ].copy()
    
    if not valid_df.empty:
        # 3. Ensure strictly string type (prevents NoneType error)
        cands = valid_df['predicted_summary'].astype(str).tolist()
        refs = valid_df['summary'].astype(str).tolist()
        
        # Calculate BERTScore
        try:
            P, R, F1 = score(cands, refs, lang="en", verbose=False)
            
            avg_f1 = F1.mean().item()
            avg_p = P.mean().item()
            
            print(f"{method_name}: F1 = {avg_f1:.4f} | Precision = {avg_p:.4f}")
            
            # Save individual CSVs
            filename = f"llm_predictions_summarisation_dynamic_{method_name}.csv"
            valid_df.to_csv(filename, index=False)
            print(f"  Saved to {filename}")
            
            summary_metrics.append({'Method': method_name, 'F1': avg_f1, 'Precision': avg_p})
            
        except Exception as e:
            print(f"Error calculating score for {method_name}: {e}")
    else:
        print(f"Skipping {method_name}: No valid predictions found after filtering.")

# Display Summary Table
if summary_metrics:
    metrics_df = pd.DataFrame(summary_metrics).sort_values('F1', ascending=False)
    print("\nComparison Table:")
    print(metrics_df)
else:
    print("No valid results to display.")

C:\Users\brand\anaconda3\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Loading DialogSum dataset from Hugging Face...
Train dataset loaded. Total records: 12460
Test dataset loaded. Total records: 1500

Evaluation set prepared: 150 items from Test set.

--- Pre-computing Indices for Retrieval ---
Loading sentence-transformer model ('all-MiniLM-L6-v2')...
Encoding Training Dialogues (Dense Index)...


Batches:   0%|          | 0/390 [00:00<?, ?it/s]

Fitting TF-IDF Vectorizer (Sparse Index)...

Starting Prediction Run using Method: MMR
  Processed 10/150 rows...
  Processed 20/150 rows...
  Processed 30/150 rows...
  Processed 40/150 rows...
  Processed 50/150 rows...
  Processed 60/150 rows...
  Processed 70/150 rows...
  Processed 80/150 rows...
  Processed 90/150 rows...
  Processed 100/150 rows...
  Processed 110/150 rows...
  Processed 120/150 rows...
  Processed 130/150 rows...
  Processed 140/150 rows...
  Processed 150/150 rows...

All predictions complete.

--- Final Evaluation (BERTScore F1) ---


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


MMR: F1 = 0.9058 | Precision = 0.8954
  Saved to llm_predictions_summarisation_dynamic_MMR.csv

Comparison Table:
  Method        F1  Precision
0    MMR  0.905834   0.895446


In [3]:
# Setup Client
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key="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" 
)


methods_to_test = ["BM25_Complex"]
final_results = {}

for method in methods_to_test:
    print(f"\n==================================================")
    print(f"Starting Prediction Run using Method: {method}")
    print(f"==================================================")
    
    current_preds = []
    
    # Iterate through the Test Set
    for i, (index, row) in enumerate(test_df.iterrows()):
        target_dialogue = row['dialogue']
        
        # 1. Retrieve Dynamic Exemplars
        try:
            shots = get_few_shots(target_dialogue, method, n_shots=3)
            # 2. Construct Prompt with these shots
            system_content = construct_dynamic_system_prompt(shots)
        except Exception as e:
            print(f"Error in retrieval: {e}")
            system_content = "You are an expert Summarizer. Output ONLY the summary."

        user_content = f"Dialogue: {target_dialogue}\nSummary:"
        
        # 3. Call LLM
        for attempt in range(3):
            try:
                response = client.chat.completions.create(
                    model="llama-3.1-8b-instant",
                    messages=[
                        {"role": "system", "content": system_content}, 
                        {"role": "user", "content": user_content}
                    ],
                    temperature=0,
                    max_tokens=60
                )
                pred = response.choices[0].message.content.strip()
                if pred:
                    current_preds.append(pred)
                    break
            except Exception as e:
                if attempt == 2: 
                    current_preds.append("error")
                    print(f"API Error: {e}")
                time.sleep(5) # Short sleep for retry
        
        time.sleep(5) # Rate limit courtesy
        
        # Print progress every 10 rows
        if (i + 1) % 10 == 0:
            print(f"  Processed {i + 1}/{len(test_df)} rows...")

    # Store results
    final_results[method] = test_df.copy()
    final_results[method]['predicted_summary'] = current_preds
    time.sleep(60)

print("\nAll predictions complete.")


# --- 5. Evaluate with BERTScore & Compare ---
print("\n--- Final Evaluation (BERTScore F1) ---")

summary_metrics = []

for method_name, result_df in final_results.items():
    # 1. Filter out explicit errors ('error' string)
    # 2. Drop rows where 'summary' or 'predicted_summary' is NaN/None
    valid_df = result_df[
        (result_df['predicted_summary'] != 'error') & 
        (result_df['summary'].notna()) & 
        (result_df['predicted_summary'].notna())
    ].copy()
    
    if not valid_df.empty:
        # 3. Ensure strictly string type (prevents NoneType error)
        cands = valid_df['predicted_summary'].astype(str).tolist()
        refs = valid_df['summary'].astype(str).tolist()
        
        # Calculate BERTScore
        try:
            P, R, F1 = score(cands, refs, lang="en", verbose=False)
            
            avg_f1 = F1.mean().item()
            avg_p = P.mean().item()
            
            print(f"{method_name}: F1 = {avg_f1:.4f} | Precision = {avg_p:.4f}")
            
            # Save individual CSVs
            filename = f"llm_predictions_summarisation_dynamic_{method_name}.csv"
            valid_df.to_csv(filename, index=False)
            print(f"  Saved to {filename}")
            
            summary_metrics.append({'Method': method_name, 'F1': avg_f1, 'Precision': avg_p})
            
        except Exception as e:
            print(f"Error calculating score for {method_name}: {e}")
    else:
        print(f"Skipping {method_name}: No valid predictions found after filtering.")

# Display Summary Table
if summary_metrics:
    metrics_df = pd.DataFrame(summary_metrics).sort_values('F1', ascending=False)
    print("\nComparison Table:")
    print(metrics_df)
else:
    print("No valid results to display.")


Starting Prediction Run using Method: BM25_Complex
  Processed 10/150 rows...
  Processed 20/150 rows...
  Processed 30/150 rows...
  Processed 40/150 rows...
  Processed 50/150 rows...
  Processed 60/150 rows...
  Processed 70/150 rows...
  Processed 80/150 rows...
  Processed 90/150 rows...
  Processed 100/150 rows...
  Processed 110/150 rows...
  Processed 120/150 rows...
  Processed 130/150 rows...
  Processed 140/150 rows...
  Processed 150/150 rows...

All predictions complete.

--- Final Evaluation (BERTScore F1) ---


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BM25_Complex: F1 = 0.9072 | Precision = 0.8964
  Saved to llm_predictions_summarisation_dynamic_BM25_Complex.csv

Comparison Table:
         Method        F1  Precision
0  BM25_Complex  0.907184   0.896411
